<a href="https://colab.research.google.com/github/xmessi08/OS-Project7/blob/main/OS_P7ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import time
import threading

REQ_FIFO = "/tmp/req_fifo"
RESP_FIFO = "/tmp/resp_fifo"

# Очистка старых файлов каналов, если они остались
for fifo in [REQ_FIFO, RESP_FIFO]:
    if os.path.exists(fifo):
        os.unlink(fifo)

# Создание FIFO
os.mkfifo(REQ_FIFO)
os.mkfifo(RESP_FIFO)

def server():
    print("[Сервер] Запущен и ожидает запросы...")
    # Открываем сервер на чтение запросов
    # Внимание: открытие FIFO на чтение блокируется, пока кто-то не откроет его на запись
    fd_req = os.open(REQ_FIFO, os.O_RDONLY)

    while True:
        data = os.read(fd_req, 1024)
        if not data:
            break
        expr = data.decode().strip()
        if expr == "EXIT":
            break
        print(f"[Сервер] Получено выражение: {expr}")

        # Вычисление (безопасный eval для простых математических операций)
        try:
            result = str(eval(expr))
        except Exception as e:
            result = f"Ошибка: {e}"

        # Отправляем ответ через второй FIFO
        fd_resp = os.open(RESP_FIFO, os.O_WRONLY)
        os.write(fd_resp, result.encode())
        os.close(fd_resp)
        time.sleep(0.1)

    os.close(fd_req)
    print("[Сервер] Остановлен.")

def client(client_id, expr):
    time.sleep(0.2 * client_id)
    print(f"[Клиент {client_id}] Отправляет: {expr}")

    # Сначала открываем ответный канал на чтение (чтобы сервер мог записать)
    fd_resp = os.open(RESP_FIFO, os.O_RDONLY)
    # Затем отправляем запрос
    fd_req = os.open(REQ_FIFO, os.O_WRONLY)
    os.write(fd_req, expr.encode())
    os.close(fd_req)

    # Читаем ответ
    resp = os.read(fd_resp, 1024).decode()
    os.close(fd_resp)
    print(f"[Клиент {client_id}] Ответ от сервера: {resp}")

# Запускаем сервер в отдельном потоке
srv_thread = threading.Thread(target=server)
srv_thread.start()

# Запускаем клиентов по очереди
expressions = ["2 + 2 * 2", "100 / 5", "10 * 10 - 5"]
client_threads = []
for i, expr in enumerate(expressions, 1):
    t = threading.Thread(target=client, args=(i, expr))
    client_threads.append(t)
    t.start()

for t in client_threads:
    t.join()

# Завершаем сервер
fd_req = os.open(REQ_FIFO, os.O_WRONLY)
os.write(fd_req, b"EXIT")
os.close(fd_req)
srv_thread.join()

# Очистка файлов FIFO
os.unlink(REQ_FIFO)
os.unlink(RESP_FIFO)
print("[Готово] FIFO каналы успешно удалены.")

[Сервер] Запущен и ожидает запросы...
[Клиент 1] Отправляет: 2 + 2 * 2
[Клиент 2] Отправляет: 100 / 5
[Клиент 3] Отправляет: 10 * 10 - 5


In [ ]:
import socket
import os
import threading
import time

SOCKET_PATH = "/tmp/unix_socket"

if os.path.exists(SOCKET_PATH):
    os.unlink(SOCKET_PATH)

def socket_server():
    server_sock = socket.socket(socket.AF_UNIX, socket.SOCK_STREAM)
    server_sock.bind(SOCKET_PATH)
    server_sock.listen(5)
    print("[Unix Сокет Сервер] Работает...")

    server_sock.settimeout(2.0) # Чтобы сервер мог завершиться по таймауту
    connections = []

    while True:
        try:
            conn, _ = server_sock.accept()
            connections.append(conn)
            t = threading.Thread(target=handle_client, args=(conn,))
            t.start()
        except socket.timeout:
            break

    server_sock.close()
    print("[Unix Сокет Сервер] Остановлен.")

def handle_client(conn):
    data = conn.recv(1024)
    if data:
        expr = data.decode().strip()
        try:
            res = str(eval(expr))
        except Exception as e:
            res = str(e)
        conn.sendall(res.encode())
    conn.close()

def socket_client(client_id, expr):
    time.sleep(0.1 * client_id)
    client_sock = socket.socket(socket.AF_UNIX, socket.SOCK_STREAM)
    client_sock.connect(SOCKET_PATH)
    client_sock.sendall(expr.encode())
    resp = client_sock.recv(1024).decode()
    print(f"[Клиент сокета {client_id}] Выражение: {expr} | Ответ: {resp}")
    client_sock.close()

# Запуск сервера
srv_thread = threading.Thread(target=socket_server)
srv_thread.start()
time.sleep(0.5)

# Запуск клиентов
client_threads = [
    threading.Thread(target=socket_client, args=(1, "50 + 50")),
    threading.Thread(target=socket_client, args=(2, "99 - 19")),
    threading.Thread(target=socket_client, args=(3, "8 * 8"))
]

for t in client_threads: t.start()
for t in client_threads: t.join()

srv_thread.join()
if os.path.exists(SOCKET_PATH):
    os.unlink(SOCKET_PATH)
print("[Готово] Unix-сокет удален.")

In [ ]:
import time
import array
from multiprocessing import shared_memory, Pipe

DATA_SIZE = 10_000_000 # 10 миллионов чисел

# Подготовка данных (массив целых чисел)
# Используем array для компактного хранения в памяти
original_array = array.array('i', range(DATA_SIZE))
bytes_data = original_array.tobytes()

# 1. Тест через Pipe
parent_conn, child_conn = Pipe()

def pipe_sender(conn, data):
    conn.send(bytes_data)
    conn.close()

import multiprocessing
p = multiprocessing.Process(target=pipe_sender, args=(child_conn, bytes_data))
start_time = time.time()
p.start()
received_data = parent_conn.recv()
p.join()
pipe_time = time.time() - start_time
print(f"Время передачи через Pipe: {pipe_time:.4f} сек")

# 2. Тест через Shared Memory
# Выделяем блок разделяемой памяти размером с наш массив в байтах
shm = shared_memory.SharedMemory(create=True, size=len(bytes_data))
# Копируем данные в разделяемую память
shm.buf[:len(bytes_data)] = bytes_data

def shm_receiver(shm_name, size):
    # Дочерний процесс подключается к существующей разделяемой памяти по имени
    existing_shm = shared_memory.SharedMemory(name=shm_name)
    # Создаем view (представление) без копирования данных
    arr = array.array('i')
    arr.frombytes(existing_shm.buf[:size])
    existing_shm.close()

start_time = time.time()
p_shm = multiprocessing.Process(target=shm_receiver, args=(shm.name, len(bytes_data)))
p_shm.start()
p_shm.join()
shm_time = time.time() - start_time
print(f"Время передачи (доступ) через Shared Memory: {shm_time:.4f} сек")

# Очистка ресурсов разделяемой памяти
shm.close()
shm.unlink()
print(f"Shared Memory быстрее Pipe примерно в {pipe_time / shm_time:.1f} раз(а).")